In [2]:
%pip install -q openai python-dotenv



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env
load_dotenv()

# Initialize the OpenAI client (automatically reads OPENAI_API_KEY)
client = OpenAI()

def call_openai(prompt, model="gpt-4o-mini", temperature=0.7, response_format=None):
    """Send a single prompt to OpenAI and return the text response."""
    kwargs = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
    }
    if response_format:
        kwargs["response_format"] = response_format

    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content.strip()

def run_multiple(prompt, n=5, model="gpt-4o-mini", temperature=0.7, max_workers=5, response_format=None):
    """Run the same prompt n times concurrently and return a list of responses."""
    def _task(_):
        return call_openai(
            prompt,
            model=model,
            temperature=temperature,
            response_format=response_format,
        )

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(_task, range(n)))
    return results

def analyze_consistency(results):
    """Inspect unique outputs and consistency percentage (0% = no repeats, 100% = all identical)."""
    counts = Counter(results)
    most_common_output, top_count = counts.most_common(1)[0]
    n = len(results)
    consistency_pct = ((top_count - 1) / (n - 1) * 100) if n > 1 else 100.0
    print(f"Total runs: {n} | Unique responses: {len(counts)} | Consistency: {consistency_pct:.1f}% (top match: {top_count}/{n})")
    return counts


# Checkpoint verification
print(call_openai("Reply with exactly: Setup successful!"))


Setup successful!


In [3]:
# Task 1: Sentiment Analysis (v1)
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

# Task 2: Product Description Generation (v1)
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

# Task 3: Data Extraction (v1)
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

for name, prompt in [
    ("Sentiment Analysis", sentiment_prompt_v1),
    ("Product Description", product_prompt_v1),
    ("Data Extraction", extraction_prompt_v1),
]:
    print(f"=== {name} ===")
    print(call_openai(prompt))
    print()


=== Sentiment Analysis ===
The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

=== Product Description ===
**Product Name:** SleekWave Wireless Mouse

**Price:** $29.99

**Product Description:**

Elevate your computing experience with the SleekWave Wireless Mouse, the perfect blend of style, comfort, and functionality. Designed with the modern user in mind, this wireless mouse offers exceptional performance at an unbeatable price.

**Key Features:**

- **Ergonomic Design:** The SleekWave is crafted to fit comfortably in your hand, reducing strain during extended use. Its smooth contours and lightweight build make it the ideal companion for both work and play.

- **Wireless Freedom:** Say goodbye to tangled cords! The SleekWave connects effortlessly to your device via a reliable 2.4GHz wireless connection, providing a seamless experience with a range of up to 33 feet. Enjoy the freedom to move without limitations.

- **Precision Tracking:** Equ

In [6]:
runs_5 = {
    "Sentiment Analysis": run_multiple(sentiment_prompt_v1, n=5),
    "Product Description": run_multiple(product_prompt_v1, n=5),
    "Data Extraction": run_multiple(extraction_prompt_v1, n=5),
}

for task, outputs in runs_5.items():
    print(f"================ {task} (5 runs) ================")
    analyze_consistency(outputs)
    for i, out in enumerate(outputs, start=1):
        print(f"--- Run {i} ---")
        print(out)
    print()


================ Sentiment Analysis (5 runs) ================
Total runs: 5 | Unique responses: 5 | Consistency: 0.0% (top match: 1/5)
--- Run 1 ---
This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
--- Run 2 ---
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
--- Run 3 ---
The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
--- Run 4 ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
--- Run 5 ---
The customer message can be classified as **Positive Feedback**.

================ Product Description (5 runs) ================
Total runs: 5 | Unique responses: 5 | Consistency: 0.0% (top match: 1/5)
--- Run 1 ---
**Product Description: Wireless Precision Mouse – $29.99**

Elevate your computing experience with our Wireless Precision Mouse, designed for seamless navigation and ultimate comfort. Priced at just $29.99, th

## Step 3: 5-Run Baseline Observations (v1)

### Metric Note: Why We Changed the Consistency Formula
- At first, 1 identical answer out of 5 showed up as **20%**, which felt misleading because it makes it look like there is some consistency when **no two answers actually matched** (and someone wouldn't know it meant "1 out of 5" unless they knew how many runs were done).
- We updated the consistency formula to `(top_count - 1) / (total_runs - 1)` so **0.0% = no repeated exact answers** and **100.0% = every answer is identical**.

### Sentiment Analysis (5 runs: 0.0% consistency, 5 unique answers out of 5)
- **What happened:** Every single try came back slightly different, even though all 5 agreed the customer was happy.
- **Differences spotted:** Some started with "This customer message..." and others with "The customer message...". Some used quotation marks, some used bold text, four gave two labels ("Positive Feedback" or "Customer Satisfaction"), and Run 5 only gave one ("Positive Feedback").

### Product Description (5 runs: 0.0% consistency, 5 unique answers out of 5)
- **What happened:** Every single try was a different long ad, and the AI made up details that weren't in the prompt.
- **Differences spotted:** It invented a different mouse name every time (Wireless Precision Mouse, SwiftConnect Wireless Mouse, SleekTech Wireless Mouse, SwiftClick Wireless Mouse), changed the headings, and made up different features and specs (like DPI numbers, battery life, or extra buttons).

### Data Extraction (5 runs: 0.0% consistency, 5 unique answers out of 5)
- **What happened:** The core facts were mostly right, but the wording and layout kept shifting so no two answers matched exactly.
- **Differences spotted:** Some added an intro line ("Here is the extracted information...") and others didn't. Bold colons moved around (`**Order Number:**` vs `**Order Number**:`), `#12345` sometimes lost its `#`, and Run 3 used "Issue: Damaged packaging" instead of "Packaging Condition: Damaged".


In [7]:
runs_10 = {
    "Sentiment Analysis": run_multiple(sentiment_prompt_v1, n=10),
    "Product Description": run_multiple(product_prompt_v1, n=10),
    "Data Extraction": run_multiple(extraction_prompt_v1, n=10),
}

for task, outputs in runs_10.items():
    print(f"================ {task} (10 runs) ================")
    analyze_consistency(outputs)
    for i, out in enumerate(outputs, start=1):
        print(f"--- Run {i} ---")
        print(out)
    print()


================ Sentiment Analysis (10 runs) ================
Total runs: 10 | Unique responses: 6 | Consistency: 22.2% (top match: 3/10)
--- Run 1 ---
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
--- Run 2 ---
The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
--- Run 3 ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
--- Run 4 ---
The customer message can be classified as **positive feedback** or **customer satisfaction**.
--- Run 5 ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
--- Run 6 ---
This customer message can be classified as **positive feedback** or **customer satisfaction**.
--- Run 7 ---
The customer message can be classified as "Positive Feedback" or "Satisfaction."
--- Run 8 ---
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
--- Run 9 ---
T

## Step 4: 10-Run Observations (v1)

### Sentiment Analysis (10 runs: 22.2% consistency, 6 unique answers out of 10, top match 3/10)
- What happened: A couple of exact wordings repeated a few times, especially "This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**." (3 times), but there were still 6 different versions overall.
- Differences spotted: "The" vs "This", quotes vs bold, lowercase vs capitalized labels (Runs 4 and 6), and "Satisfaction." instead of "Customer Satisfaction." (Run 7).

### Product Description (10 runs: 0.0% consistency, 10 unique answers out of 10, top match 1/10)
- What happened: Every single one of the 10 runs came out different, with plenty of made-up product names and specs.
- Differences spotted: Different names like SwiftClick, SleekWave, Wireless Freedom, Wireless Precision, and GlidePro, plus shifting details like range (30 ft vs 33 ft), battery life (12 months vs 18 months), and whether the AA battery is included.

### Data Extraction (10 runs: 11.1% consistency, 8 unique answers out of 10, top match 2/10)
- What happened: The core facts were mostly consistent, but tiny differences in wording and formatting kept most runs from matching.
- Differences spotted: Runs 3 and 9 were identical, while others varied on intro lines (or had none in Run 8), used "Item Number" (Run 2) or "Delivery" instead of "Delivery Speed" (Runs 7 and 10), or changed bolding (none in Run 10; colon inside asterisks in Runs 1 and 4).


In [10]:
runs_15 = {
    "Sentiment Analysis": run_multiple(sentiment_prompt_v1, n=15),
    "Product Description": run_multiple(product_prompt_v1, n=15),
    "Data Extraction": run_multiple(extraction_prompt_v1, n=15),
}

for task, outputs in runs_15.items():
    print(f"================ {task} (15 runs) ================")
    analyze_consistency(outputs)
    for i, out in enumerate(outputs, start=1):
        print(f"--- Run {i} ---")
        print(out)
    print()


================ Sentiment Analysis (15 runs) ================
Total runs: 15 | Unique responses: 8 | Consistency: 35.7% (top match: 6/15)
--- Run 1 ---
The customer message can be classified as "Positive Feedback" or "Satisfaction."
--- Run 2 ---
This customer message can be classified as **positive feedback** or **customer satisfaction**.
--- Run 3 ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
--- Run 4 ---
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
--- Run 5 ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
--- Run 6 ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
--- Run 7 ---
This customer message can be classified as **positive feedback** or **satisfaction**.
--- Run 8 ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
--- Run 9 ---
Thi

## Step 5: 15-Run Failure Analysis (v1 Baseline)

### Sentiment Analysis (15 runs: 35.7% consistency, 8 unique answers, top match 6/15)
- **vs. 10-Run Baseline:** Moved from `22.2%` (`n = 10`) to `35.7%` (`n = 15`).
- **Main Failure Patterns:** Conversational filler, dual labels (`Positive Feedback` or `Customer Satisfaction`), and casing, quote, or bolding drift.

### Product Description (15 runs: 0.0% consistency, 15 unique answers, top match 1/15)
- **vs. 10-Run Baseline:** Stayed at `0.0%` (`n = 10` and `n = 15`).
- **Main Failure Patterns:** Invented product names (*SwiftClick*, *GlidePro*, etc.), fabricated specs (2.4GHz, DPI, battery, OS), and shifting sections.

### Data Extraction (15 runs: 14.3% consistency, 12 unique answers, top match 3/15)
- **vs. 10-Run Baseline:** Moved from `11.1%` (`n = 10`) to `14.3%` (`n = 15`).
- **Main Failure Patterns:** Varying intro text, drifting field names (`Order Number` vs. `Item number`), and inconsistent `#` prefix and bolding.

### Key Takeaways
- **Sample Size (`n = 10` → `15`):** Consistency rose slightly on Sentiment (`22.2%` → `35.7%`) and Extraction (`11.1%` → `14.3%`) because larger samples reduce noise and capture the model's favored default phrasing more reliably. Over many runs, consistency converges toward the true probability of the top response rather than growing forever (and stays `0.0%` when verbatim repeats are near impossible, like Product Description).
- **Root Cause Across All Tasks:** Open-ended prompts leave wording, structure, and scope unconstrained, causing high output variance.


In [8]:
sentiment_prompt_v2 = """Classify the sentiment of the customer message below using ONLY one exact label from this list:
Positive Feedback
Neutral Feedback
Negative Feedback

Rules:
- Output ONLY the exact label.
- Do not include punctuation, quotes, bolding, or explanation.

Customer message: "I received my order quickly and everything works great!"
"""

product_prompt_v2 = """Write a short product description using ONLY the facts provided below.

Facts:
- Product: Wireless Mouse
- Price: $29.99
- Connection: Wireless

Rules:
- Do NOT invent brand names, DPI numbers, battery life, range, compatibility lists, or extra features.
- Use this exact structure:

**Wireless Mouse ($29.99)**
[One short paragraph, 2-3 sentences max, mentioning only wireless connection and the $29.99 price.]
"""

extraction_prompt_v2 = """Extract the fields below from the customer feedback and return ONLY valid JSON with these exact keys:
- "order_id" (digits only, without #)
- "order_date"
- "delivery_speed"
- "packaging_condition"

Rules:
- Output ONLY the raw JSON object. No intro text and no markdown code fences.

Customer feedback: "My order #12345 from March 15th arrived fast, though the outer box was damaged."
"""


In [9]:
runs_v2_10 = {
    "Sentiment Analysis v2": run_multiple(sentiment_prompt_v2, n=10),
    "Product Description v2": run_multiple(product_prompt_v2, n=10),
    "Data Extraction v2": run_multiple(extraction_prompt_v2, n=10),
}

for task, outputs in runs_v2_10.items():
    print(f"================ {task} (10 runs) ================")
    analyze_consistency(outputs)
    for i, out in enumerate(outputs, start=1):
        print(f"--- Run {i} ---")
        print(out)
    print()


================ Sentiment Analysis v2 (10 runs) ================
Total runs: 10 | Unique responses: 1 | Consistency: 100.0% (top match: 10/10)
--- Run 1 ---
Positive Feedback
--- Run 2 ---
Positive Feedback
--- Run 3 ---
Positive Feedback
--- Run 4 ---
Positive Feedback
--- Run 5 ---
Positive Feedback
--- Run 6 ---
Positive Feedback
--- Run 7 ---
Positive Feedback
--- Run 8 ---
Positive Feedback
--- Run 9 ---
Positive Feedback
--- Run 10 ---
Positive Feedback

================ Product Description v2 (10 runs) ================
Total runs: 10 | Unique responses: 10 | Consistency: 0.0% (top match: 1/10)
--- Run 1 ---
**Wireless Mouse ($29.99)**  
Experience the freedom of wireless connectivity with our Wireless Mouse, available for just $29.99. Say goodbye to tangled cords and enjoy seamless navigation with this convenient device. Perfect for both work and leisure, it provides an efficient solution for your computing needs.
--- Run 2 ---
**Wireless Mouse ($29.99)**  
Experience the freed

## Part 3: 10-Run Observations (v2) & v1 Comparison

### Sentiment Analysis v2 (10 runs: 100.0% consistency, 1 unique answer out of 10, top match 10/10)
- **v1 vs. v2:** Jumped from `22.2%` (`3/10`) in v1 to `100.0%` (`10/10`) in v2 (`+77.8%`).
- **What happened:** Every run returned `Positive Feedback` with no extra text, punctuation, or formatting differences, eliminating filler and dual-label hedging via a strict label list.

### Product Description v2 (10 runs: 0.0% exact-match consistency, 10 unique answers out of 10, top match 1/10)
- **v1 vs. v2:** Stayed at `0.0%` (`1/10`) exact-match consistency (`0.0%` change).
- **What happened:** Exact string consistency stayed `0.0%` because paragraph wording varied each time, but prompt constraints worked—no made-up specs (DPI, battery life, range), no invented brand names, and every run kept the `**Wireless Mouse ($29.99)**` header and short length.

### Data Extraction v2 (10 runs: 88.9% consistency, 2 unique answers out of 10, top match 9/10)
- **v1 vs. v2:** Jumped from `11.1%` (`2/10`) in v1 to `88.9%` (`9/10`) in v2 (`+77.8%`).
- **What happened:** Enforcing raw JSON with exact keys worked; all 10 runs parsed to identical data, with Run 7 differing only in 4-space vs. 2-space indentation.


In [11]:
sentiment_prompt_v3 = """Classify the sentiment of the customer message into exactly one of these labels:
- Positive Feedback
- Neutral Feedback
- Negative Feedback

Rules:
- Output ONLY the exact label.
- Do not include punctuation, explanations, or formatting.

Example:
Message: "The package arrived on time."
Output: Neutral Feedback

Message: "I received my order quickly and everything works great!"
Output:"""

product_prompt_v3 = """Write a product description using ONLY the facts provided below. Follow the exact output template word-for-word.

Facts:
- Product: Wireless Mouse
- Price: $29.99
- Key Benefit: Cable-free navigation for everyday computing

Rules:
- Do not invent brand names, technical specs, battery life, or compatibility claims.
- Use the exact structure and phrasing shown in the example.

Example Output:
**Wireless Mouse ($29.99)**
Enjoy simple, cable-free navigation for your everyday computing needs with our Wireless Mouse, priced at $29.99.

Now generate the output for the facts above:"""

extraction_prompt_v3 = """Extract order details from the customer feedback into a single-line minified JSON object.

Required keys: "order_id", "order_date", "delivery_speed", "packaging_condition"

Rules:
- Output ONLY raw, single-line minified JSON (no newlines, no indentation, no markdown code blocks).
- Strip any "#" symbol from order_id.
- Use lowercase values for delivery_speed and packaging_condition.

Example:
Feedback: "Order #99999 from June 1st was slow and the box arrived intact."
Output: {"order_id":"99999","order_date":"June 1st","delivery_speed":"slow","packaging_condition":"intact"}

Feedback: "My order #12345 from March 15th arrived fast, though the outer box was damaged."
Output:"""

runs_v3_10 = {
    "Sentiment Analysis v3": run_multiple(sentiment_prompt_v3, n=10),
    "Product Description v3": run_multiple(product_prompt_v3, n=10),
    "Data Extraction v3": run_multiple(extraction_prompt_v3, n=10),
}

for task, outputs in runs_v3_10.items():
    print(f"================ {task} (10 runs) ================")
    analyze_consistency(outputs)
    for i, out in enumerate(outputs, start=1):
        print(f"--- Run {i} ---")
        print(out)
    print()


================ Sentiment Analysis v3 (10 runs) ================
Total runs: 10 | Unique responses: 1 | Consistency: 100.0% (top match: 10/10)
--- Run 1 ---
Positive Feedback
--- Run 2 ---
Positive Feedback
--- Run 3 ---
Positive Feedback
--- Run 4 ---
Positive Feedback
--- Run 5 ---
Positive Feedback
--- Run 6 ---
Positive Feedback
--- Run 7 ---
Positive Feedback
--- Run 8 ---
Positive Feedback
--- Run 9 ---
Positive Feedback
--- Run 10 ---
Positive Feedback

================ Product Description v3 (10 runs) ================
Total runs: 10 | Unique responses: 1 | Consistency: 100.0% (top match: 10/10)
--- Run 1 ---
**Wireless Mouse ($29.99)**  
Enjoy simple, cable-free navigation for your everyday computing needs with our Wireless Mouse, priced at $29.99.
--- Run 2 ---
**Wireless Mouse ($29.99)**  
Enjoy simple, cable-free navigation for your everyday computing needs with our Wireless Mouse, priced at $29.99.
--- Run 3 ---
**Wireless Mouse ($29.99)**  
Enjoy simple, cable-free naviga

## Iteration 2: 10-Run Observations (v3) & Progression Summary

### Sentiment Analysis v3 (10 runs: 100.0% consistency, 1 unique answer out of 10, top match 10/10)
- **Progression:** `22.2%` (`v1`) → `100.0%` (`v2`) → `100.0%` (`v3`).
- **What happened:** Every run returned `Positive Feedback` cleanly with no extra text, punctuation, or formatting differences.

### Product Description v3 (10 runs: 100.0% consistency, 1 unique answer out of 10, top match 10/10)
- **Progression:** `0.0%` (`v1`) → `0.0%` exact-match (`v2`) → `100.0%` (`v3`).
- **What happened:** Combining strict fact grounding with a concrete example locked in identical output across all 10 runs without invented specs or brand names.
- **Trade-off note:** Because the example closely matched the target facts and enforced strict template adherence, the model reproduced the example's phrasing rather than generating varied creative copy—a deliberate trade-off to hit `100.0%` exact-string consistency.

### Data Extraction v3 (10 runs: 100.0% consistency, 1 unique answer out of 10, top match 10/10)
- **Progression:** `11.1%` (`v1`) → `88.9%` (`v2`) → `100.0%` (`v3`).
- **What happened:** Enforcing single-line minified JSON with a trailing completion cue eliminated the indentation drift seen in `v2` (`Run 7`), producing identical JSON on all 10 runs.

### New Technique: Completion / Few-Shot Trailing Cue (`Output:`)
- **What it is:** Providing an example (`Message: ...` / `Output: ...`) and ending the target prompt with `Output:` creates a fill-in-the-blank completion pattern.
- **Why it helps:** It cues the model to continue directly with the target value instead of adding conversational filler or reformatting.
- **Python syntax note:** Trailing `"""` right after `Output:` simply closes the Python triple-quoted string—it is not part of the prompt text sent to the model.

In [14]:
extraction_cot_prompt = """Extract order details from the customer feedback.
First write a brief step-by-step check under Reasoning: identifying order_id (without '#'), order_date, delivery_speed (lowercase), and packaging_condition (lowercase).
Then on the final line, output ONLY raw single-line minified JSON after Output: with no markdown code blocks.

Required keys: "order_id", "order_date", "delivery_speed", "packaging_condition"

Example:
Feedback: "Order #99999 from June 1st was slow and the box arrived intact."
Reasoning:
- order_id: 99999 (stripped '#')
- order_date: June 1st
- delivery_speed: slow
- packaging_condition: intact
Output: {"order_id":"99999","order_date":"June 1st","delivery_speed":"slow","packaging_condition":"intact"}

Feedback: "My order #12345 from March 15th arrived fast, though the outer box was damaged."
Reasoning:"""

runs_cot_15 = run_multiple(extraction_cot_prompt, n=15)

analyze_consistency(runs_cot_15)
print("\n--- All 15 Runs (or Unique Outputs) ---")
for i, out in enumerate(runs_cot_15, start=1):
    print(f"[Run {i}]\n{out}\n")

# Extract JSON after 'Output:' (or any line containing '{')
final_lines = [
    line.split("Output:", 1)[-1].strip()
    for out in runs_cot_15
    for line in out.splitlines()
    if "{" in line
]
print("--- Final JSON Line Consistency (15 runs) ---")
analyze_consistency(final_lines)

Total runs: 15 | Unique responses: 1 | Consistency: 100.0% (top match: 15/15)

--- All 15 Runs (or Unique Outputs) ---
[Run 1]
- order_id: 12345 (stripped '#')
- order_date: March 15th
- delivery_speed: fast
- packaging_condition: damaged
Output: {"order_id":"12345","order_date":"March 15th","delivery_speed":"fast","packaging_condition":"damaged"}

[Run 2]
- order_id: 12345 (stripped '#')
- order_date: March 15th
- delivery_speed: fast
- packaging_condition: damaged
Output: {"order_id":"12345","order_date":"March 15th","delivery_speed":"fast","packaging_condition":"damaged"}

[Run 3]
- order_id: 12345 (stripped '#')
- order_date: March 15th
- delivery_speed: fast
- packaging_condition: damaged
Output: {"order_id":"12345","order_date":"March 15th","delivery_speed":"fast","packaging_condition":"damaged"}

[Run 4]
- order_id: 12345 (stripped '#')
- order_date: March 15th
- delivery_speed: fast
- packaging_condition: damaged
Output: {"order_id":"12345","order_date":"March 15th","delivery_s

Counter({'{"order_id":"12345","order_date":"March 15th","delivery_speed":"fast","packaging_condition":"damaged"}': 15})

## Step 10: Chain-of-Thought (CoT) Data Extraction (15 Runs)

### Overall & Final JSON Consistency (15 runs: 100.0% consistency, 1 unique output out of 15, top match 15/15)
- **Full Output Consistency:** `100.0%` (`15/15` identical across both the reasoning steps and the `Output:` line).
- **Final JSON Line Consistency:** `100.0%` (`15/15` identical minified JSON: `{"order_id":"12345","order_date":"March 15th","delivery_speed":"fast","packaging_condition":"damaged"}`).

### Reasoning Step Evaluation
- **Concept clarification:** Chain-of-Thought (CoT) has the model generate intermediate reasoning steps in its output before the final answer (guided by an example format), which is distinct from writing step-by-step instructions for the model to follow internally.
- **Field & rule accuracy:** Every run correctly identified `order_id` (`12345` with `#` stripped), `order_date` (`March 15th`), `delivery_speed` (`fast`), and `packaging_condition` (`damaged`).
- **Reasoning consistency:** Because the prompt provided a concrete CoT bullet format and ended with `Reasoning:`, the reasoning steps stayed `100.0%` identical across all 15 runs instead of drifting into varied prose—though testing a trickier or edge-case extraction input would stress-test the reasoning even further.

In [16]:
product_structured_fewshot_prompt = """Write a product description using ONLY the provided facts and exact structure. Do not invent brand names, specs, battery life, or compatibility claims.

Example 1:
Facts: Product = Desk Lamp | Price = $45.00 | Benefit = Focused lighting for reading
Output:
Title: Desk Lamp ($45.00)
Summary: Provides focused lighting for reading using only the stated product details.

Example 2:
Facts: Product = Notebook | Price = $12.00 | Benefit = Daily note-taking on paper
Output:
Title: Notebook ($12.00)
Summary: Supports daily note-taking on paper using only the stated product details.

Now generate for:
Facts: Product = Wireless Mouse | Price = $29.99 | Benefit = Cable-free navigation for everyday computing
Output:"""

runs_step11_15 = run_multiple(product_structured_fewshot_prompt, n=15)

analyze_consistency(runs_step11_15)
print("\n--- All 15 Runs ---")
for i, text in enumerate(runs_step11_15, start=1):
    print(f"[Run {i}]\n{text}\n")


Total runs: 15 | Unique responses: 2 | Consistency: 92.9% (top match: 14/15)

--- All 15 Runs ---
[Run 1]
Title: Wireless Mouse ($29.99)  
Summary: Enables cable-free navigation for everyday computing using only the stated product details.

[Run 2]
Title: Wireless Mouse ($29.99)  
Summary: Enables cable-free navigation for everyday computing using only the stated product details.

[Run 3]
Title: Wireless Mouse ($29.99)  
Summary: Enables cable-free navigation for everyday computing using only the stated product details.

[Run 4]
Title: Wireless Mouse ($29.99)  
Summary: Enables cable-free navigation for everyday computing using only the stated product details.

[Run 5]
Title: Wireless Mouse ($29.99)  
Summary: Enables cable-free navigation for everyday computing using only the stated product details.

[Run 6]
Title: Wireless Mouse ($29.99)  
Summary: Enables cable-free navigation for everyday computing using only the stated product details.

[Run 7]
Title: Wireless Mouse ($29.99)  
Sum

## Step 11: Few-Shot + Structured Format Product Description (15 Runs)

### Consistency & Output Summary (15 runs: 92.9% consistency, 2 unique outputs out of 15, top match 14/15)
- **Consistency Metric:** `92.9%` (`((14 - 1) / (15 - 1) * 100)`), with `14/15` runs matching the top output and `2` unique responses overall.
- **Dominant Output (14/15 runs — Runs 1–9, 11–15):**
  `Title: Wireless Mouse ($29.99)` / `Summary: Enables cable-free navigation for everyday computing using only the stated product details.`
- **Variant Output (1/15 runs — Run 10):** Changed a single verb to `Offers cable-free navigation...` while keeping the exact same `Title:` / `Summary:` structure and remainder of the sentence.

### Format, Few-Shot & Generation Observations
- **Structure & grounding held:** Every run preserved the `Title:` and `Summary:` layout, fact grounding, and trailing clause without inventing specs or brand names.
- **Controlled adaptation vs. verbatim copying:** Unlike a single verbatim example that gets copied word-for-word, providing two examples (`Provides` and `Supports`) alongside a structured format encouraged the model to pick a fitting verb (`Enables` most of the time, `Offers` once) rather than copying a line verbatim—trading a tiny `7.1%` consistency dip (`92.9%` vs. `100.0%`) for slightly more natural adaptation.


In [17]:
product_structured_fewshot_prompt = """Write a concise retail product description following the exact Title/Summary format and using only the facts given for that item. Do not invent brand names or extra specs not listed in the facts.

Example 1:
Facts: Product = Matte Black Metal Desk Lamp | Price = $45.00 | Material = Powder-coated metal | Style = Minimalist task lamp | Benefit = Focused lighting for reading
Output:
Title: Matte Black Metal Desk Lamp ($45.00)
Summary: Minimalist task lamp in a powder-coated matte black metal finish that casts focused light right over your book or desk for comfortable reading.

Example 2:
Facts: Product = Linen Hardcover Notebook | Price = $12.00 | Material = Textured linen cover with smooth cream lined paper | Benefit = Daily note-taking on paper
Output:
Title: Linen Hardcover Notebook ($12.00)
Summary: Sturdy linen-bound hardcover with smooth cream lined pages built for daily note-taking, lists, and quick sketches on paper.

Now generate for:
Facts: Product = Wireless Mouse | Price = $29.99 | Benefit = Cable-free navigation for everyday computing
Output:"""

runs_step11_15 = run_multiple(product_structured_fewshot_prompt, n=15)

analyze_consistency(runs_step11_15)
print("\n--- All 15 Runs ---")
for i, text in enumerate(runs_step11_15, start=1):
    print(f"[Run {i}]\n{text}\n")


Total runs: 15 | Unique responses: 10 | Consistency: 21.4% (top match: 4/15)

--- All 15 Runs ---
[Run 1]
Title: Wireless Mouse ($29.99)  
Summary: Enjoy cable-free navigation with this wireless mouse, designed for everyday computing with ease and convenience.

[Run 2]
Title: Wireless Mouse ($29.99)  
Summary: Enjoy cable-free navigation for everyday computing with this sleek and functional wireless mouse, designed for effortless control and convenience.

[Run 3]
Title: Wireless Mouse ($29.99)  
Summary: Enjoy cable-free navigation with this wireless mouse, designed for seamless everyday computing and effortless control.

[Run 4]
Title: Wireless Mouse ($29.99)  
Summary: Enjoy cable-free navigation with this wireless mouse, designed for everyday computing tasks and effortless use.

[Run 5]
Title: Wireless Mouse ($29.99)  
Summary: Enjoy cable-free navigation with this wireless mouse, designed for seamless everyday computing and effortless control.

[Run 6]
Title: Wireless Mouse ($29.99

## Step 11 (Revised): Rich Retail Few-Shot Product Description (15 Runs)

### Consistency & Output Summary (15 runs: 21.4% consistency, 10 unique outputs out of 15, top match 4/15)
- **Consistency Metric:** `21.4%` (`((4 - 1) / (15 - 1) * 100)`), with `10` unique responses and the most frequent string appearing `4/15` times (Runs 3, 5, 7, 13).
- **Dominant Pattern (15/15 runs):** All runs preserved the `Title: Wireless Mouse ($29.99)` header, `Summary:` label, and the opening phrase `Enjoy cable-free navigation...`.

### Key Observations
- **Consistency drop from few-shot & rich retail examples:** Moving from a single simple example to multi-example, attribute-rich retail descriptions caused exact-string consistency to drop sharply (`21.4%`), as more descriptive phrasing gave the model more lexical choices across runs.
- **High phrasing/synonym variance:** Differences were driven by shifting between `everyday computing`, `everyday computing tasks`, and `seamless everyday computing`, alongside varying endings (`effortless control`, `ease and convenience`, `effortless use`, `enhanced mobility`, `enhanced productivity`).
- **Soft promotional embellishment:** While the model avoided inventing technical specs or brand names, it added soft marketing adjectives (`sleek`, `functional`, `efficient`, `mobility`, `productivity`) beyond the literal input benefit (`Cable-free navigation for everyday computing`).
- **Determinism vs. retail polish trade-off:** Richer, more natural retail prose improves human readability but reduces exact-string repeatability unless constrained further or paired with post-processing/lower variance settings.


In [20]:
# --- STEP 12A: Customer Complaint Reply + Human Escalation ---
complaint_escalation_prompt_template = """You are a customer support assistant. Reply concisely and politely without promising refunds or policy exceptions not stated in the message.
If human escalation is needed, include the escalation contact email escalations@example-support.test in your reply.
Then output a single routing line at the end in the exact format:
Escalate: YES or Escalate: NO | Reason: <short reason>

Customer message:
"{message}"
"""

complaint_variations = [
    # Clear escalation (safety / strong frustration / refund demand)
    "My toaster started smoking and smelled like burnt plastic! I want a full refund immediately and to speak to a manager.",
    # Borderline / moderate frustration (damaged item, calm tone)
    "Hi, the glass jar arrived with a crack on the lid. Can you help me get a replacement?",
    # Routine question (no escalation needed)
    "Hello, do you know if the medium size in navy blue will be back in stock this week?",
]

print("=== STEP 12A: COMPLAINT & ESCALATION VARIATIONS (ALL RUNS) ===")
for idx, msg in enumerate(complaint_variations, start=1):
    prompt = complaint_escalation_prompt_template.format(message=msg)
    runs = run_multiple(prompt, n=5)
    print(f"\n[Variation {idx}] {msg}")
    analyze_consistency(runs)
    for r_num, out in enumerate(runs, start=1):
        print(f"  [Run {r_num}]\n{out}\n")


# --- STEP 12B: Store Locator Reply (Unrelated Sample Address) ---
store_reply_prompt_template = """You are a store locator assistant. Using ONLY the provided store directory entry below, answer the customer's question concisely and include the exact store name and street address when recommending a store. Do not invent hours, stock, or policies not listed.

Available Store:
- Name: Harborview Market & Goods
- Address: 742 Maple Avenue, Suite 2B, Portland, OR 97205
- Hours: Mon-Sat: 9AM - 7PM

Customer question:
"{question}"
"""

store_reply_variations = [
    "Where is the store located and what are its hours?",
    "Can you share the street address and opening hours?",
    "Do you have a branch in Denver?",
]

print("\n=== STEP 12B: STORE LOCATOR REPLY VARIATIONS (ALL RUNS) ===")
for idx, q in enumerate(store_reply_variations, start=1):
    prompt = store_reply_prompt_template.format(question=q)
    runs = run_multiple(prompt, n=5)
    print(f"\n[Variation {idx}] {q}")
    analyze_consistency(runs)
    for r_num, out in enumerate(runs, start=1):
        print(f"  [Run {r_num}]\n{out}\n")


=== STEP 12A: COMPLAINT & ESCALATION VARIATIONS (ALL RUNS) ===

[Variation 1] My toaster started smoking and smelled like burnt plastic! I want a full refund immediately and to speak to a manager.
Total runs: 5 | Unique responses: 5 | Consistency: 0.0% (top match: 1/5)
  [Run 1]
I'm sorry to hear about the issue with your toaster. For assistance regarding your concern, please reach out to our escalations team at escalations@example-support.test. They will be able to help you further.

Escalate: YES | Reason: Customer requests to speak to a manager.

  [Run 2]
Thank you for reaching out. I'm sorry to hear about the issue with your toaster. For any concerns regarding refunds or to escalate your matter to a manager, please contact our escalation team at escalations@example-support.test. 

Escalate: YES | Reason: Customer requests to speak to a manager.

  [Run 3]
I'm sorry to hear about your experience with the toaster. For concerns regarding refunds and to speak with a manager, please re

## Step 12A & 12B (Zero-Shot / Non-CoT): Task Variations (5 Runs Each)

### Step 12A: Customer Complaint & Escalation (5 Runs Each)
- **Variation 1 (Smoking Toaster / Refund / Manager):** `0.0%` consistency (`5` unique, top match `1/5`); `5/5` routed to `Escalate: YES` and included `escalations@example-support.test`.
- **Variation 2 (Cracked Jar Lid / Replacement):** `0.0%` consistency (`5` unique, top match `1/5`); `4/5` routed to `Escalate: YES` and `1/5` to `Escalate: NO`.
- **Variation 3 (Navy Blue Medium Restock Inquiry):** `0.0%` consistency (`5` unique, top match `1/5`); `4/5` routed to `Escalate: NO` and `1/5` to `Escalate: YES`.
- **Observations:**
  - Free-form replies and varying `Reason:` clauses produced `0.0%` exact-string consistency across every variation.
  - Escalation logic was rock-solid for clear high-severity/manager requests (Variation 1) but inconsistent on moderate damage (Variation 2) and routine stock questions (Variation 3).
  - The escalation email occasionally appeared even on `Escalate: NO` replies (Variation 3, Runs 2 and 5), showing a need for stricter conditional rules.

### Step 12B: Store Locator Reply (5 Runs Each)
- **Variation 1 (Location & Hours):** `50.0%` consistency (`3` unique, top match `3/5` — Runs 1, 2, 5).
- **Variation 2 (Street Address & Opening Hours):** `0.0%` consistency (`5` unique, top match `1/5`).
- **Variation 3 (Branch in Denver?):** `25.0%` consistency (`3` unique, top match `2/5` — Runs 2, 5).
- **Observations:**
  - All 15 runs across all three variations stayed strictly grounded in the provided directory entry (`Harborview Market & Goods`, `742 Maple Avenue, Suite 2B, Portland, OR 97205`, `Mon–Sat`, `9 AM–7 PM`).
  - Exact-string variance came from minor synonyms, optional commas, and time spacing (`9 AM` vs. `9AM`) rather than factual errors or hallucinations.


In [21]:
# --- STEP 12A (1-Shot + CoT): Customer Complaint Reply + Human Escalation ---
complaint_1shot_cot_template = """You are a customer support assistant. Reply concisely and politely without promising refunds or policy exceptions not stated in the message.
If human escalation is needed, include the escalation contact email escalations@example-support.test in your reply.

Follow this exact output structure:
Reasoning: <brief step-by-step evaluation of safety/severity, frustration/refund request, and whether human escalation is needed>
Reply: <concise customer response>
Escalate: YES or Escalate: NO | Reason: <short reason>

Example:
Customer message:
"My order hasn't arrived after two weeks and I'm really frustrated—can someone look into this?"
Output:
Reasoning: The customer reports a delayed shipment and moderate frustration, which benefits from human follow-up to check carrier status.
Reply: I'm sorry for the delay with your order. For further assistance with tracking your shipment, please reach out to our escalation team at escalations@example-support.test.
Escalate: YES | Reason: Delayed order requiring support follow-up.

Now respond to:
Customer message:
"{message}"
Output:"""

complaint_variations = [
    # Clear escalation (safety / strong frustration / refund demand)
    "My toaster started smoking and smelled like burnt plastic! I want a full refund immediately and to speak to a manager.",
    # Borderline / moderate frustration (damaged item, calm tone)
    "Hi, the glass jar arrived with a crack on the lid. Can you help me get a replacement?",
    # Routine question (no escalation needed)
    "Hello, do you know if the medium size in navy blue will be back in stock this week?",
]

print("=== STEP 12A (1-SHOT + CoT): COMPLAINT & ESCALATION VARIATIONS (ALL RUNS) ===")
for idx, msg in enumerate(complaint_variations, start=1):
    prompt = complaint_1shot_cot_template.format(message=msg)
    runs = run_multiple(prompt, n=5)
    print(f"\n[Variation {idx}] {msg}")
    analyze_consistency(runs)
    for r_num, out in enumerate(runs, start=1):
        print(f"  [Run {r_num}]\n{out}\n")


# --- STEP 12B (1-Shot + CoT): Store Locator Reply ---
store_1shot_cot_template = """You are a store locator assistant. Using ONLY the provided store directory entry below, answer the customer's question concisely and include the exact store name and street address when recommending a store. Do not invent hours, stock, or policies not listed.

Available Store:
- Name: Harborview Market & Goods
- Address: 742 Maple Avenue, Suite 2B, Portland, OR 97205
- Hours: Mon-Sat: 9AM - 7PM

Follow this exact output structure:
Reasoning: <brief check of what the customer asked and which directory facts apply>
Reply: <concise store locator answer>

Example:
Customer question:
"What are your hours?"
Output:
Reasoning: The customer is asking for store hours, which are listed as Monday to Saturday, 9 AM to 7 PM, and I will also include the store name and address for clarity.
Reply: Harborview Market & Goods (742 Maple Avenue, Suite 2B, Portland, OR 97205) is open Monday to Saturday from 9 AM to 7 PM.

Now respond to:
Customer question:
"{question}"
Output:"""

store_reply_variations


=== STEP 12A (1-SHOT + CoT): COMPLAINT & ESCALATION VARIATIONS (ALL RUNS) ===

[Variation 1] My toaster started smoking and smelled like burnt plastic! I want a full refund immediately and to speak to a manager.
Total runs: 5 | Unique responses: 5 | Consistency: 0.0% (top match: 1/5)
  [Run 1]
Reasoning: The customer reports a serious safety issue with the toaster, indicating a possible fire hazard, which requires immediate attention. The frustration level is high due to the request for a refund and manager involvement.
Reply: I'm sorry to hear about the issue with your toaster. Please ensure it is unplugged and safely set aside. For further assistance and to escalate this matter, you can contact our escalation team at escalations@example-support.test.
Escalate: YES | Reason: Safety issue with the product requiring urgent attention.

  [Run 2]
Reasoning: The customer reports a safety issue with their toaster, indicating a potentially severe situation. Their request for a refund and esc

['Where is the store located and what are its hours?',
 'Can you share the street address and opening hours?',
 'Do you have a branch in Denver?']

## Step 12A (1-Shot + CoT): Customer Complaint & Escalation Variations (5 Runs Each)

### Consistency Summary
- **Variation 1 (Safety / Refund / Manager):** `0.0%` exact-string consistency (`5/5` unique outputs, top match `1/5`), with `5/5` routing to `Escalate: YES`.
- **Variation 2 (Damaged Jar / Replacement):** `0.0%` exact-string consistency (`5/5` unique outputs, top match `1/5`), with `5/5` routing to `Escalate: YES`.
- **Variation 3 (Restock Inquiry):** `0.0%` exact-string consistency (`5/5` unique outputs, top match `1/5`), with split routing (`3/5` `Escalate: NO`, `2/5` `Escalate: YES`).

### Comparison vs. Previous Run (Zero-Shot / Non-CoT)
- **Exact-string consistency (`0.0%` vs. `0.0%`):** Unchanged across all three variations (`5/5` unique outputs each), confirming that free-form prose—now expanded with `Reasoning:` and `Reply:` labels—prevents exact-string matches regardless of 1-shot + CoT.
- **Variation 1 routing (`5/5 YES` vs. `5/5 YES`):** Matched on escalation outcome, while 1-shot + CoT explicitly surfaced the fire/safety hazard and added practical safety guidance (e.g., unplugging the toaster).
- **Variation 2 routing (`5/5 YES` vs. `4/5 YES`, `1/5 NO`):** Improved routing agreement on the cracked jar replacement request by eliminating the lone `NO` outlier from the previous run.
- **Variation 3 routing (`3/5 NO`, `2/5 YES` vs. `4/5 NO`, `1/5 YES`):** Slightly worse false-positive escalation rate on the routine navy blue restock question (rising from `1/5` to `2/5` `Escalate: YES`), with `escalations@example-support.test` still appearing whenever the model chose to escalate.

### Key Observations
- **Free-form CoT & prose drive `0.0%` exact-string match:** Adding `Reasoning:`, free-form `Reply:` sentences, and custom `Reason:` clauses introduces natural wording variance on every single run.
- **Routing stabilized on clear/moderate complaints, still drifts on routine queries:** Variations 1 and 2 reached `5/5` `Escalate: YES`, whereas Variation 3 still occasionally triggered `Escalate: YES` for a routine stock check.
- **Prompt tuning needed:** To make routing deterministic and prevent unnecessary escalations on routine questions, add explicit inclusion/exclusion rules for `Escalate: YES` vs. `Escalate: NO` (and separate routing/structured fields from free-form prose if exact-string or programmatic consistency is required).


In [22]:
# --- STEP 12A (1-Shot, No CoT, Tuned Routing): Customer Complaint Reply + Human Escalation ---
complaint_1shot_no_cot_template = """You are a customer support assistant. Reply concisely and politely without promising refunds or policy exceptions not stated in the message.

Routing rules:
- Output Escalate: YES for safety hazards, damaged/defective/missing items, refund/cancellation/manager requests, or strong frustration, and include escalations@example-support.test in your reply.
- Output Escalate: NO for routine inquiries (such as stock availability, sizing, colors, or general product/store info) with no complaint or issue, and do NOT include escalations@example-support.test.

Follow this exact output structure:
Reply: <concise customer response>
Escalate: YES or Escalate: NO | Reason: <short reason>

Example:
Customer message:
"Do you carry this jacket in large or know when it restocks?"
Output:
Reply: Thank you for reaching out! I don't have live details on sizing or restock dates right now, so please check back on the product page for updates.
Escalate: NO | Reason: Routine stock and sizing question with no issue or complaint.

Now respond to:
Customer message:
"{message}"
Output:"""

complaint_variations = [
    # Clear escalation (safety / strong frustration / refund demand)
    "My toaster started smoking and smelled like burnt plastic! I want a full refund immediately and to speak to a manager.",
    # Borderline / moderate frustration (damaged item, calm tone)
    "Hi, the glass jar arrived with a crack on the lid. Can you help me get a replacement?",
    # Routine question (no escalation needed)
    "Hello, do you know if the medium size in navy blue will be back in stock this week?",
]

print("=== STEP 12A (1-SHOT, NO CoT, TUNED ROUTING): COMPLAINT & ESCALATION VARIATIONS (ALL RUNS) ===")
for idx, msg in enumerate(complaint_variations, start=1):
    prompt = complaint_1shot_no_cot_template.format(message=msg)
    runs = run_multiple(prompt, n=5)
    print(f"\n[Variation {idx}] {msg}")
    analyze_consistency(runs)
    for r_num, out in enumerate(runs, start=1):
        print(f"  [Run {r_num}]\n{out}\n")


=== STEP 12A (1-SHOT, NO CoT, TUNED ROUTING): COMPLAINT & ESCALATION VARIATIONS (ALL RUNS) ===

[Variation 1] My toaster started smoking and smelled like burnt plastic! I want a full refund immediately and to speak to a manager.
Total runs: 5 | Unique responses: 5 | Consistency: 0.0% (top match: 1/5)
  [Run 1]
Reply: I'm very sorry to hear about the issue with your toaster. For safety concerns and to assist you further, I will escalate your request to our support team.  
Escalate: YES | Reason: Safety hazard and request for manager assistance.

  [Run 2]
Reply: I'm sorry to hear about your toaster issue! For safety concerns like this, I will escalate your request to ensure it is addressed properly. 
Escalate: YES | Reason: Safety hazard and request for manager.

  [Run 3]
Reply: I'm sorry to hear about your toaster! For safety concerns and to address your request, I will escalate this to our management team for further assistance.  
Escalate: YES | Reason: Safety hazard and request for

## Step 12A (1-Shot, No CoT, Tuned Routing): Customer Complaint & Escalation Variations (5 Runs Each)

### Consistency & Routing Summary
- **Variation 1 (Safety / Refund / Manager):** `0.0%` exact-string consistency (`5/5` unique outputs, top match `1/5`), with `5/5` routing to `Escalate: YES`.
- **Variation 2 (Damaged Jar / Replacement):** `0.0%` exact-string consistency (`5/5` unique outputs, top match `1/5`), with `5/5` routing to `Escalate: YES`.
- **Variation 3 (Restock Inquiry):** `0.0%` exact-string consistency (`5/5` unique outputs, top match `1/5`), with `5/5` routing to `Escalate: NO`.

### Comparison vs. Previous Runs (Zero-Shot & 1-Shot + CoT)
- **Exact-string consistency (`0.0%` across all three runs):** Unchanged (`5/5` unique outputs per variation), confirming that open-ended `Reply:` prose and custom `Reason:` text prevent exact-string matches even without `Reasoning:`.
- **Routine query routing (`5/5 NO` vs. `4/5 NO, 1/5 YES` zero-shot and `3/5 NO, 2/5 YES` 1-shot + CoT):** Eliminated false-positive escalations on the routine restock inquiry and kept `escalations@example-support.test` off all `Escalate: NO` replies.
- **Complaint routing (`5/5 YES` on Variations 1 & 2):** Matched the `5/5 YES` stability seen in 1-shot + CoT on both clear safety/refund complaints and damaged-item replacement requests.

### Key Observations
- **Explicit routine-ask rules + a `NO` example sharply improved routing reliability:** Carving out stock/sizing/general inquiries as `Escalate: NO` resolved the ambiguity that previously caused occasional `YES` outputs on Variation 3.
- **Routing consistency vs. exact-string consistency:** While `Escalate: YES/NO` agreement reached `100%` (`5/5`) on every variation, free-form phrasing in `Reply:` and `Reason:` kept exact-string consistency at `0.0%`.
- **Minor tuning opportunity:** Escalation email usage appeared in `3/5` runs of Variation 2 and `0/5` runs of Variation 1; clarifying when the assistant escalates internally vs. directs the user to `escalations@example-support.test` will make replies even more consistent.


In [23]:
# --- STEP 13: ALL v3 PROMPTS & 15-RUN EVALUATION IN ONE BLOCK ---

sentiment_v3_template = """Classify the sentiment of the customer text as strictly one word: POSITIVE, NEGATIVE, or NEUTRAL.
Do not include explanations, punctuation, or extra text.

Text:
"{text}"
Sentiment:"""

extraction_v3_template = """Extract order details from the text and output ONLY a single valid JSON object with exact keys:
{{"item": string_or_null, "quantity": integer_or_null, "color": string_or_null, "issue": string_or_null}}
Use null for any field not explicitly stated. Do not add markdown fences or extra text.

Text:
"{text}"
JSON:"""

product_v3_template = """Write a concise retail product description using ONLY the provided facts. Do not invent brand names, features, or specs not in the facts.

Follow this exact format:
Title: <Product> (<Price>)
Summary: <one concise retail sentence grounded in the given facts>

Facts:
{facts}
Output:"""

complaint_v3_template = """You are a customer support assistant. Reply concisely and politely without promising refunds or policy exceptions not stated in the message.

Routing rules:
- Output Escalate: YES for safety hazards, damaged/defective/missing items, refund/cancellation/manager requests, or strong frustration, and include escalations@example-support.test in your reply.
- Output Escalate: NO for routine inquiries (such as stock availability, sizing, colors, or general product/store info) with no complaint or issue, and do NOT include escalations@example-support.test.

Follow this exact output structure:
Reply: <concise customer response>
Escalate: YES or Escalate: NO | Reason: <short reason>

Example:
Customer message:
"Do you carry this jacket in large or know when it restocks?"
Output:
Reply: Thank you for reaching out! I don't have live details on sizing or restock dates right now, so please check back on the product page for updates.
Escalate: NO | Reason: Routine stock and sizing question with no issue or complaint.

Customer message:
"{message}"
Output:"""

store_v3_template = """You are a store locator assistant. Using ONLY the provided store directory entry below, answer the customer's question concisely and include the exact store name and street address when recommending a store. Do not invent hours, stock, branches, or policies not listed.

Available Store:
- Name: Harborview Market & Goods
- Address: 742 Maple Avenue, Suite 2B, Portland, OR 97205
- Hours: Mon-Sat: 9AM - 7PM

Customer question:
"{question}"
Reply:"""

test_inputs_v3 = {
    "sentiment": sentiment_v3_template.format(
        text="The delivery arrived a day early and the packaging was spotless."
    ),
    "extraction": extraction_v3_template.format(
        text="I ordered 2 ceramic mugs in matte white, and one arrived with a chipped handle."
    ),
    "product": product_v3_template.format(
        facts="Product = Wireless Mouse | Price = $29.99 | Benefit = Cable-free navigation for everyday computing"
    ),
    "complaint": complaint_v3_template.format(
        message="Hi, the glass jar arrived with a crack on the lid. Can you help me get a replacement?"
    ),
    "store": store_v3_template.format(
        question="Where is the store located and what are its hours?"
    ),
}

step13_results = {}
for task_name, prompt in test_inputs_v3.items():
    print(f"\n================ STEP 13 (15 RUNS): {task_name.upper()} v3 ================")
    runs = run_multiple(prompt, n=15)
    step13_results[task_name] = runs
    analyze_consistency(runs)
    for i, out in enumerate(runs, start=1):
        print(f"[Run {i}]\n{out}\n")



================ STEP 13 (15 RUNS): SENTIMENT v3 ================
Total runs: 15 | Unique responses: 1 | Consistency: 100.0% (top match: 15/15)
[Run 1]
POSITIVE

[Run 2]
POSITIVE

[Run 3]
POSITIVE

[Run 4]
POSITIVE

[Run 5]
POSITIVE

[Run 6]
POSITIVE

[Run 7]
POSITIVE

[Run 8]
POSITIVE

[Run 9]
POSITIVE

[Run 10]
POSITIVE

[Run 11]
POSITIVE

[Run 12]
POSITIVE

[Run 13]
POSITIVE

[Run 14]
POSITIVE

[Run 15]
POSITIVE


================ STEP 13 (15 RUNS): EXTRACTION v3 ================
Total runs: 15 | Unique responses: 4 | Consistency: 64.3% (top match: 10/15)
[Run 1]
{"item":"ceramic mug","quantity":2,"color":"matte white","issue":"chipped handle"}

[Run 2]
{"item":"ceramic mug","quantity":2,"color":"matte white","issue":"chipped handle"}

[Run 3]
{"item":"ceramic mug","quantity":2,"color":"matte white","issue":"one arrived with a chipped handle"}

[Run 4]
{"item":"ceramic mug","quantity":2,"color":"matte white","issue":"chipped handle"}

[Run 5]
{"item":"ceramic mug","quantity":2,"col

## Observations: Step 13 Last Batch (15 Runs Each, v3 Prompts)

- **Sentiment (`100.0%`, `15/15`):** Completely deterministic `POSITIVE` output across all 15 runs with zero extra text or formatting drift.
- **Extraction (`64.3%`, `10/15`):** Valid JSON and exact schema on all 15 runs; minor variance came from singular vs. plural `"item"` (`"ceramic mug"` vs. `"ceramic mugs"`) and short vs. full-phrase `"issue"` (`"chipped handle"` vs. `"one arrived with a chipped handle"`).
- **Product (`50.0%`, `8/15`):** Grounded in facts and locked to `Title:` / `Summary:`, with small wording swaps (`"Enjoy"` vs. `"Experience"`, optional adjectives) and trailing spaces keeping exact-string match at `50.0%`.
- **Complaint (`0.0%` exact-string, `1/15`):** Routing was `100%` (`15/15` `Escalate: YES`), while free-form `Reply:` and `Reason:` wording made all 15 full strings unique and `escalations@example-support.test` appeared in `11/15` runs.
- **Store (`71.4%`, `11/15`):** Accurate name, synthetic address (`742 Maple Avenue, Suite 2B, Portland, OR 97205`), and hours across all 15 runs, with slight phrasing and `9 AM` vs. `9AM` spacing differences.

---

## Full Evaluation & Suite Observations (Including Recorded Personal Observations)

### Personal Observations Recorded Across This Conversation
- **Consistency Metric Definition:** Measured repeatability using `((top_count - 1) / (n - 1) * 100)`.
- **1-Shot Verbatim Copying vs. Few-Shot Variety:** One-shot examples increased consistency in product descriptions by encouraging verbatim copying, whereas few-shot examples were less consistent because there wasn't a single verbatim example to mirror.
- **Natural Language Lowers Consistency:** Anything involving natural language reduced consistency due to its creative/variable nature; full consistency across everything was achieved only with one-shot setups where outputs didn't drift into varied prose.
- **Dropping `Reason` (CoT + 1-Shot → 1-Shot Only):** Removing the `Reason` / CoT step on the newer prompts (such as customer complaint) improved consistency by cutting out a natural-language field.
- **Tuning `NO` / Neutral Routing:** Needed to improve routing accuracy for `NO` / neutral/routine cases because the model initially seemed overly focused on detecting `YES` escalations.

### Additional Things Seen in the Data / Runs
- **Routing vs. Exact-String Split in Step 13 Complaint (`15/15 YES`, `0.0%` exact match):** Even when the `Escalate: YES` decision was 100% consistent across all 15 runs, tiny wording tweaks in `Reply:` and `Reason:` (plus trailing spaces on several lines) still made all 15 full outputs unique.
- **Inconsistent Escalation Email Presence (`11/15` runs):** On the `Escalate: YES` cracked-jar runs, `escalations@example-support.test` showed up in 11 of 15 runs, while the remaining 4 asked for order details or a photo or phrased it as an internal escalation without giving the address.
- **Small Format/Spacing Drift in Store & Product:** In Store v3 (`71.4%`), 11/15 matched the main string while 4 varied slightly (`"The store you're/you’re/you are looking for..."` or `"9AM"` vs. `"9 AM"`). In Extraction v3 (`64.3%`), 10/15 matched identically while `"ceramic mug"` vs. `"ceramic mugs"` and `"chipped handle"` vs. `"one arrived with a chipped handle"` accounted for the remaining runs.
